# LeetCode #38 — Count and Say: mã hóa độ dài loạt (Run-Length Encoding)

**Mức độ:** Medium · **Chủ đề:** String

Skeleton trên LeetCode:

```python
class Solution:
    def countAndSay(self, n: int) -> str:
```

Mục tiêu của notebook là nắm chắc kỹ thuật **gom nhóm các ký tự liên tiếp giống nhau** (Run-Length Encoding — RLE) bằng con trỏ, một thao tác chuỗi xuất hiện rất thường xuyên. Bài này cũng là ví dụ tốt về **lặp một phép biến đổi** nhiều lần.

# 1. Bài toán là gì?

Dãy **"đếm và nói" (count-and-say)** được định nghĩa:

- `countAndSay(1) = "1"`
- `countAndSay(n)` = **đọc to** `countAndSay(n - 1)`: chia chuỗi thành các **nhóm ký tự giống nhau liên tiếp**, với mỗi nhóm nói "**số lượng** rồi **chữ số**".

Ví dụ đọc `"3322251"`:

```text
33 | 222 | 5 | 1
"hai số 3", "ba số 2", "một số 5", "một số 1"
-> "23" + "32" + "15" + "11" = "23321511"
```

**Constraints (theo đề):** `1 <= n <= 30`

---

## Các phần tử đầu của dãy

```text
n = 1:  "1"
n = 2:  "11"       (một số 1)
n = 3:  "21"       (hai số 1)
n = 4:  "1211"     (một số 2, một số 1)
n = 5:  "111221"   (một số 1, một số 2, hai số 1)
n = 6:  "312211"
```

## Ví dụ

```text
n = 4  ->  "1211"
n = 1  ->  "1"
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Chuỗi (string) trong Python
2. Hai con trỏ (two pointers)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Chuỗi (string) trong Python

### Chuỗi là bất biến

`str` **không sửa được** tại chỗ: `s[0] = "x"` gây lỗi. Mọi "sửa" đều tạo chuỗi mới. Vì vậy:

- Ghép nhiều mảnh: gom vào **list** rồi `"".join(list)` (`O(n)`), tránh `s += ...` trong vòng lặp (có thể `O(n²)`).
- Cần sửa từng ký tự: `chars = list(s)`, sửa, rồi `"".join(chars)`.

In [1]:
s_str = "hello"
try:
    s_str[0] = "H"
except TypeError as e:
    print("Sửa trực tiếp:", e)
chars_str = list(s_str); chars_str[0] = "H"
print("qua list:", "".join(chars_str))
manh = []
for i in range(5):
    manh.append(str(i))
print("join:", "-".join(manh))

Sửa trực tiếp: 'str' object does not support item assignment
qua list: Hello
join: 0-1-2-3-4


### Cắt chuỗi (slicing) và các thao tác hay dùng

| Biểu thức | Ý nghĩa |
|---|---|
| `s[i:j]` | ký tự từ `i` tới `j−1` (tạo chuỗi mới, `O(j−i)`) |
| `s[::-1]` | chuỗi đảo ngược |
| `s.split()` | tách theo khoảng trắng (bỏ khoảng trắng thừa) |
| `s.split(",")` | tách theo dấu phẩy (giữ phần rỗng) |
| `s.strip()` | bỏ khoảng trắng hai đầu |
| `s.startswith(p)`, `s.find(p)` | kiểm tra tiền tố, tìm vị trí |
| `c.isdigit()`, `c.isalpha()`, `c.isalnum()`, `c.isspace()` | phân loại ký tự |
| `c.lower()`, `c.upper()` | đổi hoa/thường |

In [2]:
t_str = "  Xin chào   thế giới  "
print(repr(t_str.split()), "|", repr(t_str.strip()), "|", repr("a,,b".split(",")))
print("đảo:", "abcdef"[::-1], "| cắt [1:4]:", "abcdef"[1:4], "| 'abc'.find('c') =", "abc".find("c"))
print([(c, c.isdigit(), c.isalpha()) for c in "a1 "])

['Xin', 'chào', 'thế', 'giới'] | 'Xin chào   thế giới' | ['a', '', 'b']
đảo: fedcba | cắt [1:4]: bcd | 'abc'.find('c') = 2
[('a', False, True), ('1', True, False), (' ', False, False)]


### Ký tự và mã số: `ord` / `chr`

Mỗi ký tự có một mã số (Unicode). `ord("a") = 97`, `chr(97) = "a"`. Mẹo: chữ số `c` → giá trị `ord(c) - ord("0")`; chữ thường `c` → chỉ số 0..25 là `ord(c) - ord("a")`.

In [3]:
print("ord('a') =", ord("a"), "| chr(98) =", chr(98))
print("'7' → 7:", ord("7") - ord("0"), "| 'd' → chỉ số:", ord("d") - ord("a"))
dem26 = [0] * 26
for c in "banana":
    dem26[ord(c) - ord("a")] += 1
print("đếm 26 chữ cái của 'banana' (a, b, n):", dem26[0], dem26[1], dem26[13])

ord('a') = 97 | chr(98) = b
'7' → 7: 7 | 'd' → chỉ số: 3
đếm 26 chữ cái của 'banana' (a, b, n): 3 1 2


### Gom nhóm ký tự liên tiếp

`itertools.groupby` gom các phần tử **liên tiếp bằng nhau** — tiện cho mã hóa độ dài loạt (run-length): `"aaabcc" → 3a 1b 2c`. Tự viết bằng hai con trỏ: `i` ở đầu nhóm, `j` chạy tới khi ký tự khác.

In [4]:
from itertools import groupby
print([(k, len(list(g))) for k, g in groupby("aaabcc")])

def run_length(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(f"{j - i}{s[i]}")
        i = j
    return "".join(out)
print("run_length('aaabcc') =", run_length("aaabcc"))

[('a', 3), ('b', 1), ('c', 2)]
run_length('aaabcc') = 3a1b2c


Lỗi hay gặp: `split(" ")` sinh phần rỗng khi có nhiều dấu cách; so sánh chuỗi số `"10" < "9"` là **True** (so từ điển, không phải số); nối chuỗi trong vòng lặp lớn; quên chuỗi rỗng.

## 2.2 Hai con trỏ (two pointers)

### Ý tưởng

Dùng **hai chỉ số** di chuyển trên mảng/chuỗi theo quy tắc, để thay vòng lặp lồng `O(n²)` bằng một lượt `O(n)`. Hai kiểu chính:

**1. Hai đầu tiến vào nhau** (thường trên mảng đã sắp xếp):

```text
[1, 2, 4, 7, 11, 15], tìm hai số có tổng 15
 L                 R     1 + 15 = 16 > 15 → R lùi (cần nhỏ hơn)
 L             R         1 + 11 = 12 < 15 → L tiến (cần lớn hơn)
    L          R         2 + 11 = 13 < 15 → L tiến
       L       R         4 + 11 = 15 ✓
```

Vì sao được phép bỏ? Khi `a[L] + a[R] > target`, mọi cặp `(L', R)` với `L' > L` còn lớn hơn → `R` không thể là đáp án, bỏ `R`.

**2. Cùng chiều — con trỏ đọc và con trỏ ghi** (sửa mảng tại chỗ):

```text
xóa mọi số 0, giữ thứ tự: [0, 1, 0, 3, 12]
read đi qua từng phần tử; gặp số khác 0 thì chép vào vị trí write, write += 1
→ [1, 3, 12, ...]
```

In [5]:
def tong_hai_so_da_sap_xep(a, target):
    L, R = 0, len(a) - 1
    while L < R:
        s = a[L] + a[R]
        print(f"   L={L} R={R}: {a[L]} + {a[R]} = {s}")
        if s == target:
            return L, R
        if s < target:
            L += 1
        else:
            R -= 1
    return None

print("→", tong_hai_so_da_sap_xep([1, 2, 4, 7, 11, 15], 15))

def bo_so_khong(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    return a[:write]

print("bỏ số 0:", bo_so_khong([0, 1, 0, 3, 12]))

def la_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True
print("'racecar' palindrome?", la_palindrome("racecar"), "| 'abca'?", la_palindrome("abca"))

   L=0 R=5: 1 + 15 = 16
   L=0 R=4: 1 + 11 = 12
   L=1 R=4: 2 + 11 = 13
   L=2 R=4: 4 + 11 = 15
→ (2, 4)
bỏ số 0: [1, 3, 12]
'racecar' palindrome? True | 'abca'? False


### Đảo ngược một đoạn tại chỗ

Hai con trỏ ở hai đầu, đổi chỗ rồi tiến vào trong — dùng rất nhiều (đảo mảng, xoay mảng, đảo từ).

In [6]:
def dao_doan(a, i, j):
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1; j -= 1

arr_tp = [1, 2, 3, 4, 5, 6]
dao_doan(arr_tp, 1, 4)
print("đảo đoạn [1..4]:", arr_tp)

đảo đoạn [1..4]: [1, 5, 4, 3, 2, 6]


### Lỗi hay gặp

- Quên **sắp xếp** trước khi dùng hai đầu cho bài tổng.
- Điều kiện `L <= R` thay vì `L < R` → dùng một phần tử hai lần.
- Không chứng minh được vì sao bỏ một đầu là an toàn → dễ bỏ mất đáp án.
- Con trỏ ghi vượt con trỏ đọc (không thể xảy ra nếu `write` chỉ tăng khi ghi).

## 2.3 Áp dụng vào bài này

Mỗi số hạng là **mã hóa độ dài loạt** (run-length) của số hạng trước: gom các chữ số **liên tiếp bằng nhau**, ghi "số lượng + chữ số". Gom nhóm bằng **hai con trỏ** `i` (đầu nhóm) và `j` (chạy tới hết nhóm) hoặc `itertools.groupby`.

# 3. Nền tảng: gom nhóm ký tự liên tiếp

Đây là kỹ thuật **Run-Length Encoding (RLE)**. Ta duyệt chuỗi bằng con trỏ `i`; với mỗi vị trí bắt đầu nhóm, dùng con trỏ `j` chạy tới hết nhóm:

```python
i = 0
while i < len(s):
    j = i
    while j < len(s) and s[j] == s[i]:
        j += 1
    # nhóm là s[i:j], gồm (j - i) ký tự s[i]
    i = j
```

In [7]:
from typing import List

def groups(s: str):
    result = []
    i = 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        result.append((s[i], j - i))
        i = j
    return result

print(groups("3322251"))
print(groups("1"))
print(groups("111221"))

[('3', 2), ('2', 3), ('5', 1), ('1', 1)]
[('1', 1)]
[('1', 3), ('2', 2), ('1', 1)]


Hai vòng `while` lồng nhau nhưng tổng cộng chỉ `O(len(s))`: `j` đi qua mỗi ký tự đúng một lần, và `i` nhảy thẳng tới `j`.

## "Đọc to" một chuỗi

Ghép mỗi nhóm thành `str(count) + char`:

In [8]:
def say(s: str) -> str:
    return "".join(str(count) + ch for ch, count in groups(s))

print(say("3322251"))
print(say("1"), say("11"), say("21"), say("1211"))

23321511
11 21 1211 111221


# 4. Lời giải chính: lặp phép "đọc" `n - 1` lần

```text
s = "1"
lặp n - 1 lần: s = say(s)
return s
```

Viết gộp lại thành một hàm, dùng list để ghép chuỗi (tránh `+=` trên chuỗi dài):

In [9]:
def count_and_say(n: int) -> str:
    s = "1"
    for _ in range(n - 1):
        parts = []
        i = 0
        while i < len(s):
            j = i
            while j < len(s) and s[j] == s[i]:
                j += 1
            parts.append(str(j - i))
            parts.append(s[i])
            i = j
        s = "".join(parts)
    return s

for n in range(1, 9):
    print(f"n = {n}: {count_and_say(n)}")

n = 1: 1
n = 2: 11
n = 3: 21
n = 4: 1211
n = 5: 111221
n = 6: 312211
n = 7: 13112221
n = 8: 1113213211


# 5. Mô phỏng từng bước: đọc `"111221"`

| i | Nhóm | j sau vòng trong | Số lượng | Nói |
|---|---|---|---|---|
| 0 | `111` | 3 | 3 | `"31"` |
| 3 | `22` | 5 | 2 | `"22"` |
| 5 | `1` | 6 | 1 | `"11"` |

Ghép: `"31" + "22" + "11" = "312211"` = `countAndSay(6)`.

In [10]:
def trace_say(s: str) -> str:
    parts = []
    i = 0
    print(f"Đọc {s!r}:")
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        print(f"  i = {i}: nhóm {s[i:j]!r} có {j - i} chữ '{s[i]}' -> nói {str(j - i) + s[i]!r}; nhảy i = {j}")
        parts.append(str(j - i) + s[i])
        i = j
    result = "".join(parts)
    print(f"  => {result!r}")
    return result

_ = trace_say("111221")
_ = trace_say("3322251")

Đọc '111221':
  i = 0: nhóm '111' có 3 chữ '1' -> nói '31'; nhảy i = 3
  i = 3: nhóm '22' có 2 chữ '2' -> nói '22'; nhảy i = 5
  i = 5: nhóm '1' có 1 chữ '1' -> nói '11'; nhảy i = 6
  => '312211'
Đọc '3322251':
  i = 0: nhóm '33' có 2 chữ '3' -> nói '23'; nhảy i = 2
  i = 2: nhóm '222' có 3 chữ '2' -> nói '32'; nhảy i = 5
  i = 5: nhóm '5' có 1 chữ '5' -> nói '15'; nhảy i = 6
  i = 6: nhóm '1' có 1 chữ '1' -> nói '11'; nhảy i = 7
  => '23321511'


# 6. Cách viết khác: `itertools.groupby`

`groupby(s)` gom các phần tử **liên tiếp bằng nhau**, trả về cặp `(key, iterator của nhóm)`.

In [11]:
from itertools import groupby

for key, grp in groupby("3322251"):
    print(key, list(grp))

def count_and_say_groupby(n: int) -> str:
    s = "1"
    for _ in range(n - 1):
        s = "".join(str(len(list(grp))) + key for key, grp in groupby(s))
    return s

print(count_and_say_groupby(6))

3 ['3', '3']
2 ['2', '2', '2']
5 ['5']
1 ['1']
312211


**Lưu ý:** `groupby` chỉ gom phần tử **liên tiếp**. `groupby("aba")` cho 3 nhóm `a`, `b`, `a` — không gộp hai chữ `a`. Đây đúng là điều ta cần cho bài này (khác với `Counter`, đếm tổng thể).

# 7. Phiên bản đúng format LeetCode

In [12]:
class Solution:
    def countAndSay(self, n: int) -> str:
        s = "1"
        for _ in range(n - 1):
            parts = []
            i = 0
            while i < len(s):
                j = i
                while j < len(s) and s[j] == s[i]:
                    j += 1
                parts.append(str(j - i))
                parts.append(s[i])
                i = j
            s = "".join(parts)
        return s

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<12} PASS  {actual}")


sol = Solution()
check("n = 1", sol.countAndSay(1), "1")
check("n = 2", sol.countAndSay(2), "11")
check("n = 3", sol.countAndSay(3), "21")
check("n = 4", sol.countAndSay(4), "1211")
check("n = 5", sol.countAndSay(5), "111221")
check("n = 6", sol.countAndSay(6), "312211")
check("n = 10", sol.countAndSay(10), "13211311123113112211")

for n in range(1, 31):
    assert sol.countAndSay(n) == count_and_say_groupby(n)
print("n = 1..30: hai cách khớp nhau")

n = 1        PASS  1
n = 2        PASS  11
n = 3        PASS  21
n = 4        PASS  1211
n = 5        PASS  111221
n = 6        PASS  312211
n = 10       PASS  13211311123113112211
n = 1..30: hai cách khớp nhau


# 8. Vì sao đúng? Độ phức tạp?

**Đúng:** code là bản dịch trực tiếp định nghĩa. Phần cốt lõi là vòng gom nhóm: `j` dừng ở ký tự **đầu tiên khác** `s[i]` (hoặc cuối chuỗi), nên `s[i:j]` là một nhóm **cực đại** (không thể kéo dài thêm), và các nhóm nối tiếp nhau phủ kín chuỗi.

**Độ dài tăng thế nào?** Xem độ dài các phần tử:

In [14]:
lengths = [len(sol.countAndSay(n)) for n in range(1, 31)]
print(lengths)
print("tỉ lệ len(n) / len(n-1) ở cuối:", [round(lengths[i] / lengths[i - 1], 3) for i in range(25, 30)])

[1, 2, 2, 4, 6, 6, 8, 10, 14, 20, 26, 34, 46, 62, 78, 102, 134, 176, 226, 302, 408, 528, 678, 904, 1182, 1540, 2012, 2606, 3410, 4462]


tỉ lệ len(n) / len(n-1) ở cuối: [1.303, 1.306, 1.295, 1.309, 1.309]


Độ dài tăng khoảng **1.3 lần** mỗi bước (hằng số Conway ≈ 1.3036). Với `n = 30` độ dài khoảng 4462 ký tự.

Gọi `L_n` là độ dài phần tử thứ `n`:

```text
Time:  O(L_1 + L_2 + ... + L_n) = O(L_n)   (tổng cấp số nhân, bị chặn bởi hằng số × L_n)
Space: O(L_n)
```

Vì `L_n` tăng theo hàm mũ (`≈ 1.3^n`), với `n <= 30` vẫn rất nhỏ.

**Thú vị:** dãy này không bao giờ chứa chữ số lớn hơn `3` — không có nhóm nào dài 4 trở lên.

In [15]:
print(all(ch in "123" for n in range(1, 31) for ch in sol.countAndSay(n)))

True

# 9. Những lỗi dễ mắc

## Lỗi 1: Đếm tổng số lần xuất hiện thay vì nhóm liên tiếp

Dùng `Counter` sẽ gộp những ký tự giống nhau nhưng **không liền kề**:

In [16]:
from collections import Counter

def wrong_counter(s: str) -> str:
    return "".join(str(cnt) + ch for ch, cnt in Counter(s).items())

print("Sai :", wrong_counter("1211"), "| Đúng:", say("1211"))

Sai : 3112 | Đúng: 111221


## Lỗi 2: Quên nhóm cuối cùng

Cách viết "so với ký tự trước, khi khác thì ghi nhóm" rất hay quên ghi **nhóm cuối** sau vòng lặp:

In [17]:
def wrong_forget_last(s: str) -> str:
    parts = []
    count = 1
    for k in range(1, len(s)):
        if s[k] == s[k - 1]:
            count += 1
        else:
            parts.append(str(count) + s[k - 1])
            count = 1
    # thiếu: parts.append(str(count) + s[-1])
    return "".join(parts)

print("Sai :", repr(wrong_forget_last("111221")), "| Đúng:", repr(say("111221")))

Sai : '3122' | Đúng: '312211'


Cách viết hai con trỏ `i`, `j` ở mục 3 tránh được lỗi này một cách tự nhiên.

## Lỗi 3: Viết "chữ số rồi số lượng"

Đúng thứ tự là **số lượng trước**, chữ số sau: `"222"` → `"32"`, không phải `"23"`.

## Lỗi 4: Lặp `n` lần thay vì `n - 1` lần

Bắt đầu từ `"1"` (đã là phần tử thứ 1), chỉ cần biến đổi `n - 1` lần.

# 10. Mẹo nhận diện pattern

**Gom nhóm liên tiếp (run-length):**

- "nhóm ký tự liên tiếp giống nhau", "nén chuỗi", "đếm số đoạn"
- Mẫu hai con trỏ `i`, `j` hoặc `itertools.groupby`.

Bài liên quan: **#443 String Compression**, **#1446 Consecutive Characters**, **#830 Positions of Large Groups**, **#271 Encode and Decode Strings**.

## Template

```python
i = 0
while i < n:
    j = i
    while j < n and s[j] == s[i]:
        j += 1
    xử lý nhóm s[i:j]
    i = j
```

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Tính `countAndSay(7)` và `countAndSay(8)` từ `countAndSay(6) = "312211"`.

## Bài 2 — Nhóm dài nhất (LeetCode #1446)

Trả về độ dài của **nhóm ký tự giống nhau liên tiếp dài nhất**.

```python
max_power("leetcode")    -> 2   ("ee")
max_power("abbcccddddeeeeedcba") -> 5
```

In [18]:
def exercise_2_max_power(s: str) -> int:
    # TODO: dùng mẫu hai con trỏ i, j
    pass


# assert exercise_2_max_power("leetcode") == 2
# assert exercise_2_max_power("abbcccddddeeeeedcba") == 5
# assert exercise_2_max_power("a") == 1

## Bài 3 — Điền chỗ trống

In [19]:
def exercise_3_count_and_say(n: int) -> str:
    s = "1"
    for _ in range(n - 1):
        parts = []
        i = 0
        # TODO: gom nhóm, thêm "số lượng" rồi "chữ số" vào parts
        s = "".join(parts)
    return s


# assert exercise_3_count_and_say(1) == "1"
# assert exercise_3_count_and_say(4) == "1211"
# assert exercise_3_count_and_say(6) == "312211"

## Bài 4 — Biến thể: giải mã RLE

Chuỗi mã hóa gồm các cặp "số lượng (có thể nhiều chữ số) + ký tự chữ cái". Giải mã:

```python
rle_decode("3a2b1c")  -> "aaabbc"
rle_decode("12x")     -> "xxxxxxxxxxxx"
rle_decode("")        -> ""
```

In [20]:
def exercise_4_rle_decode(s: str) -> str:
    # TODO
    pass


# assert exercise_4_rle_decode("3a2b1c") == "aaabbc"
# assert exercise_4_rle_decode("12x") == "x" * 12
# assert exercise_4_rle_decode("") == ""

## Bài 5 — Mini challenge: String Compression (LeetCode #443)

Nén list ký tự **tại chỗ**: mỗi nhóm thay bằng ký tự, theo sau là số lượng (nếu số lượng > 1, viết từng chữ số). Trả về **độ dài mới**; phần đầu của list chứa kết quả.

```python
chars = ["a","a","b","b","c","c","c"] -> 6, chars[:6] == ["a","2","b","2","c","3"]
chars = ["a"]                          -> 1, chars[:1] == ["a"]
chars = ["a"] + ["b"] * 12             -> 4, chars[:4] == ["a","b","1","2"]
```

Yêu cầu bộ nhớ phụ `O(1)`: dùng con trỏ `write` ghi đè lên đầu list.

In [21]:
def exercise_5_compress(chars: List[str]) -> int:
    # TODO
    pass


# c = ["a","a","b","b","c","c","c"]; assert exercise_5_compress(c) == 6 and c[:6] == ["a","2","b","2","c","3"]
# c = ["a"]; assert exercise_5_compress(c) == 1 and c[:1] == ["a"]
# c = ["a"] + ["b"] * 12; assert exercise_5_compress(c) == 4 and c[:4] == ["a","b","1","2"]

# 12. Gợi ý

**Bài 1**
- Mức 1: chia `"312211"` thành nhóm: `3 | 1 | 22 | 11`.

**Bài 2**
- Mức 1: với mỗi nhóm `s[i:j]`, cập nhật `best = max(best, j - i)`.

**Bài 4**
- Mức 1: đọc **số nhiều chữ số**: `num = num * 10 + int(ch)` (giống bài atoi).
- Mức 2: gặp chữ cái → thêm `ch * num`, đặt lại `num = 0`.

**Bài 5**
- Mức 1: hai con trỏ đọc `i`, `j` như mục 3, và con trỏ ghi `write`.
- Mức 2: ghi `chars[write] = chars[i]`; nếu `j - i > 1`, ghi từng chữ số của `str(j - i)`.
- Mức 3: con trỏ ghi không bao giờ vượt con trỏ đọc, nên ghi đè an toàn.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [22]:
s7 = trace_say("312211")
s8 = trace_say(s7)

Đọc '312211':
  i = 0: nhóm '3' có 1 chữ '3' -> nói '13'; nhảy i = 1
  i = 1: nhóm '1' có 1 chữ '1' -> nói '11'; nhảy i = 2
  i = 2: nhóm '22' có 2 chữ '2' -> nói '22'; nhảy i = 4
  i = 4: nhóm '11' có 2 chữ '1' -> nói '21'; nhảy i = 6
  => '13112221'
Đọc '13112221':
  i = 0: nhóm '1' có 1 chữ '1' -> nói '11'; nhảy i = 1
  i = 1: nhóm '3' có 1 chữ '3' -> nói '13'; nhảy i = 2
  i = 2: nhóm '11' có 2 chữ '1' -> nói '21'; nhảy i = 4
  i = 4: nhóm '222' có 3 chữ '2' -> nói '32'; nhảy i = 7
  i = 7: nhóm '1' có 1 chữ '1' -> nói '11'; nhảy i = 8
  => '1113213211'


## Đáp án Bài 2

In [23]:
def answer_2_max_power(s: str) -> int:
    best = 0
    i = 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        best = max(best, j - i)
        i = j
    return best

assert answer_2_max_power("leetcode") == 2
assert answer_2_max_power("abbcccddddeeeeedcba") == 5
assert answer_2_max_power("a") == 1
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [24]:
def answer_3_count_and_say(n: int) -> str:
    s = "1"
    for _ in range(n - 1):
        parts = []
        i = 0
        while i < len(s):
            j = i
            while j < len(s) and s[j] == s[i]:
                j += 1
            parts.append(str(j - i))
            parts.append(s[i])
            i = j
        s = "".join(parts)
    return s

assert answer_3_count_and_say(1) == "1"
assert answer_3_count_and_say(4) == "1211"
assert answer_3_count_and_say(6) == "312211"
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [25]:
def answer_4_rle_decode(s: str) -> str:
    out = []
    num = 0
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        else:
            out.append(ch * num)
            num = 0
    return "".join(out)

assert answer_4_rle_decode("3a2b1c") == "aaabbc"
assert answer_4_rle_decode("12x") == "x" * 12
assert answer_4_rle_decode("") == ""
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [26]:
def answer_5_compress(chars: List[str]) -> int:
    write = 0
    i = 0
    n = len(chars)
    while i < n:
        j = i
        while j < n and chars[j] == chars[i]:
            j += 1
        chars[write] = chars[i]
        write += 1
        if j - i > 1:
            for d in str(j - i):
                chars[write] = d
                write += 1
        i = j
    return write

c = ["a","a","b","b","c","c","c"]; assert answer_5_compress(c) == 6 and c[:6] == ["a","2","b","2","c","3"]
c = ["a"]; assert answer_5_compress(c) == 1 and c[:1] == ["a"]
c = ["a"] + ["b"] * 12; assert answer_5_compress(c) == 4 and c[:4] == ["a","b","1","2"]
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Count and Say = lặp `n - 1` lần phép "đọc to". Phép đọc = **gom nhóm ký tự liên tiếp** bằng hai con trỏ `i`, `j`, rồi ghi "số lượng + chữ số".

---

## Checklist trước khi nộp

- [ ] Gom nhóm **liên tiếp** (không dùng `Counter`)?
- [ ] Không quên nhóm cuối?
- [ ] Ghi **số lượng trước**, chữ số sau?
- [ ] Lặp `n - 1` lần?
- [ ] Dùng list + `"".join` để ghép chuỗi?